# Expanded Exercise (R version): Multiple Group Comparisons (VeryAnts Sales)
## One-way ANOVA + Post-hoc Tests with Correction, Effect Sizes & Simulation

This is the **R skeleton** version. Follow the markdown instructions and complete the TODO cells.
Use the solution notebook to verify or see advanced R approaches (TukeyHSD, pairwise.t.test with p.adjust, replicate simulation, etc.).

**Packages needed:** `tidyverse`, `ggplot2`, `car` (for Levene), `stats` (base).


## Flowchart of the Desired Analysis Outcome (Multiple Comparisons)
```mermaid
flowchart TD
    Start[Start: Load Data & EDA] --> Assumptions{Check Assumptions<br/>Normality per group + Homogeneity}
    Assumptions -->|OK| Omnibus[One-way ANOVA (aov)]<br/>(overall test for any difference)
    Omnibus -->|Significant| PostHoc[Post-hoc: TukeyHSD or pairwise.t.test(p.adjust)]
    Omnibus -->|Not Significant| Stop[No further pairwise tests]
    PostHoc --> EffectSize[Compute Cohen's d + CI for significant pairs]
    EffectSize --> Interpret[Interpret: Which stores differ?<br/>Practical + Business impact]
    Interpret --> Audience[Tailor Reporting to Audience<br/>Execs vs Analysts vs Non-technical]
    Audience --> Conclusion[Conclusion following data analysis report structure]
    Conclusion --> Simulation[Monte Carlo Simulation<br/>Modify params → see FWER & power]
    Simulation --> End[End]
```
**Note:** Same recommended workflow as the Python version. Use this flowchart in your reports.


## 1. Setup, Libraries, Data Loading & EDA

**Instructions:**
1. Load `tidyverse` and `ggplot2`
2. Read `veryants.csv`
3. Print counts and mean sales per store
4. Create a boxplot of Sale by Store using `ggplot`
5. Comment on which store appears highest/lowest from the plot


In [ ]:
# TODO: Install if needed (run once)
# install.packages(c("tidyverse", "ggplot2", "car"))

library(tidyverse)
library(ggplot2)

# TODO: Load data
veryants <- read_csv("veryants.csv")

# TODO: Inspect
print(table(veryants$Store))
print(veryants %>% group_by(Store) %>% summarise(mean_sale = mean(Sale), sd_sale = sd(Sale), n = n()))

# TODO: Boxplot
ggplot(veryants, aes(x = Store, y = Sale, fill = Store)) +
  geom_boxplot(alpha = 0.7) +
  labs(title = "Sales Distribution by VeryAnts Store", y = "Sale (USD)") +
  theme_minimal()


## 2. Assumption Checks

**Instructions:**
1. Create Q-Q plots for each store (use `qqnorm` + `qqline` or ggplot2 stat_qq)
2. Run `shapiro.test()` on each group
3. Run Levene’s test (`car::leveneTest(Sale ~ Store, data = veryants)`)
4. Decide if ANOVA / t-tests are appropriate


In [ ]:
# TODO: Q-Q plots
par(mfrow = c(1,3))
qqnorm(veryants$Sale[veryants$Store == "A"], main = "Store A"); qqline(veryants$Sale[veryants$Store == "A"], col = "red")
qqnorm(veryants$Sale[veryants$Store == "B"], main = "Store B"); qqline(veryants$Sale[veryants$Store == "B"], col = "red")
qqnorm(veryants$Sale[veryants$Store == "C"], main = "Store C"); qqline(veryants$Sale[veryants$Store == "C"], col = "red")
par(mfrow = c(1,1))

# TODO: Shapiro tests
for (s in c("A","B","C")) {
  cat(sprintf("Shapiro Store %s: p = %.4f\n", s, shapiro.test(veryants$Sale[veryants$Store == s])$p.value))
}

# TODO: Levene (requires car)
# library(car)
# leveneTest(Sale ~ Store, data = veryants)


## 3. One-way ANOVA (Omnibus Test)

**Instructions:**
1. Run `aov(Sale ~ Store, data = veryants)`
2. Use `summary()` on the aov object and interpret the p-value
3. Why start with ANOVA instead of three separate t-tests?


In [ ]:
# TODO: One-way ANOVA
anova_model <- aov(Sale ~ Store, data = veryants)
summary(anova_model)


## 4. Post-hoc Pairwise Comparisons with Correction

**Instructions:**
1. Run `TukeyHSD(anova_model)` — this is the recommended method in R
2. (Alternative) Use `pairwise.t.test(veryants$Sale, veryants$Store, p.adjust.method = "bonferroni")`
3. Compare raw p-values vs corrected p-values
4. Decide which pairs are significantly different after correction


In [ ]:
# TODO: Tukey HSD (recommended)
tukey_res <- TukeyHSD(anova_model)
print(tukey_res)

# TODO: Bonferroni via pairwise.t.test
pairwise.t.test(veryants$Sale, veryants$Store, p.adjust.method = "bonferroni")


## 5. Effect Sizes (Cohen's d)

**Instructions:**
Write a small function or calculate manually for the pairs that remain significant after correction.
Formula is the same as in Python: pooled SD version.


In [ ]:
# TODO: Cohen's d function (example for two groups)
cohens_d <- function(g1, g2) {
  n1 <- length(g1); n2 <- length(g2)
  pooled_sd <- sqrt( ((n1-1)*var(g1) + (n2-1)*var(g2)) / (n1 + n2 - 2) )
  (mean(g2) - mean(g1)) / pooled_sd
}

a <- veryants$Sale[veryants$Store == "A"]
b <- veryants$Sale[veryants$Store == "B"]
c <- veryants$Sale[veryants$Store == "C"]

cat("Cohen's d (B - A):", round(cohens_d(a, b), 3), "\n")
cat("Cohen's d (C - A):", round(cohens_d(a, c), 3), "\n")
cat("Cohen's d (B - C):", round(cohens_d(c, b), 3), "\n")


## 6. More Practice Exercises (R)

1. Try `p.adjust.method = "holm"` instead of Bonferroni.
2. Interpret the Tukey output: look at `p adj`, `lwr`, `upr`, and `diff`.
3. Business interpretation: Which store should the company investigate further?
4. What if the sample sizes were unbalanced?


## 7. Simulation Section (Modify Parameters & Re-run)

**Goal:** See the impact of multiple testing correction on false positive rate and power.
Change `true_means`, `n_per_group`, or `apply_correction` and observe the results.


In [ ]:
set.seed(42)

# === MODIFIABLE PARAMETERS ===
true_means <- c(58, 65, 62)   # change to c(60,60,60) for null (no differences)
sigma <- 15
n_per_group <- 150
n_simulations <- 500
alpha <- 0.05
apply_correction <- TRUE

# === Simulation with replicate ===
results <- replicate(n_simulations, {
  g1 <- rnorm(n_per_group, true_means[1], sigma)
  g2 <- rnorm(n_per_group, true_means[2], sigma)
  g3 <- rnorm(n_per_group, true_means[3], sigma)
  
  p12 <- t.test(g1, g2, var.equal = TRUE)$p.value
  p13 <- t.test(g1, g3, var.equal = TRUE)$p.value
  p23 <- t.test(g2, g3, var.equal = TRUE)$p.value
  pvals <- c(p12, p13, p23)
  
  if (apply_correction) pvals <- pmin(pvals * 3, 1)
  
  any(pvals < alpha)
})

fwer_or_power <- mean(results)
label <- if (all(true_means == true_means[1])) "Family-wise error rate" else "Power (at least one significant pair)"
cat(sprintf("%s: %.3f\n", label, fwer_or_power))
cat("Try apply_correction = FALSE with equal means to see ~14% FWER.\n")


## 8. Conclusion & Audience-Aware Reporting (Your Turn in R)

Write a Conclusion section following the data analysis report structure.
Create versions tailored to executives, technical supervisors, and non-technical audiences (same guidance as the Python version).
